# Brent · `decision_date` (web: "Decision date")

**Goal:** understand the data, so that a chat bot can answer any question about it
more easily and correctly.

Brent only (golden rule): every query carries the location-based Brent filter.
Data comes from the **live API**.

Workflow: add a step here → discuss the findings → decide whether it belongs in
`knowledge/brent.md` → add it.

In [1]:
from pathlib import Path

import httpx
import pandas as pd
from dotenv import dotenv_values

PROJECT_ROOT = next(p for p in [Path.cwd(), *Path.cwd().parents] if (p / "pyproject.toml").exists())
env = dotenv_values(PROJECT_ROOT / ".env")

BASE_URL = "https://planningdata.london.gov.uk/api-guest"
HEADERS = {"X-API-AllowRequest": env["DATAHUB_ALLOW_REQUEST"]}


def brent_filter() -> dict:
    # Location-based scope: borough is Brent, or borough is missing and Brent decided it.
    return {
        "bool": {
            "should": [
                {"term": {"borough.raw": "Brent"}},
                {"bool": {
                    "filter": [{"term": {"lpa_name.raw": "Brent"}}],
                    "must_not": [{"exists": {"field": "borough.raw"}}],
                }},
            ],
            "minimum_should_match": 1,
        }
    }


def search(body: dict) -> dict:
    response = httpx.post(f"{BASE_URL}/applications/_search", headers=HEADERS, json=body, timeout=60)
    response.raise_for_status()
    return response.json()

## Step 1 · Nulls and values that are not dates

Fetch `decision_date` for every Brent application (paged with `search_after`, sorted by
`id`, because there are more than 10,000). Then count:
- **null**: no value at all;
- **empty or blank**: a string with nothing in it;
- **not a date**: a value that doesn't parse as `DD/MM/YYYY` (the format the schema
  and the index mapping give).

In [2]:
rows, after = [], None
while True:
    body = {"size": 5000, "query": brent_filter(), "_source": ["id", "decision_date"], "sort": [{"id": "asc"}]}
    if after:
        body["search_after"] = after
    page = search(body)["hits"]["hits"]
    if not page:
        break
    rows += [h["_source"] for h in page]
    after = page[-1]["sort"]

df = pd.DataFrame(rows).reindex(columns=["id", "decision_date"])
print("Brent applications:", len(df))

Brent applications: 36952


In [3]:
value = df["decision_date"]
is_null = value.isna()
is_blank = value.notna() & (value.astype(str).str.strip() == "")
parsed = pd.to_datetime(value, format="%d/%m/%Y", errors="coerce")
not_a_date = value.notna() & ~is_blank & parsed.isna()

summary = pd.Series({
    "null": int(is_null.sum()),
    "empty or blank": int(is_blank.sum()),
    "not a date": int(not_a_date.sum()),
    "valid DD/MM/YYYY date": int(parsed.notna().sum()),
})
summary["total"] = summary.sum()
display(summary.to_frame("count").assign(**{"share_%": lambda t: (100 * t["count"] / len(df)).round(2)}))

print("Values that are not dates:")
df[not_a_date]

,count,share_%
null,995,2.69
empty or blank,0,0.00
not a date,0,0.00
valid DD/MM/YYYY date,35957,97.31
total,36952,100.00


Values that are not dates:


,id,decision_date


### Findings · Step 1

Live API, 2026-11-09, 36,952 Brent applications:

| | Count | Share |
|---|---|---|
| null | 995 | 2.69% |
| empty or blank | 0 | 0% |
| not a date | 0 | 0% |
| valid `DD/MM/YYYY` date | 35,957 | 97.31% |

- **No weird values:** every `decision_date` that is filled in is a real `DD/MM/YYYY`
  date. This fits the index mapping, which stores the field as type `date` with format
  `dd/MM/yyyy`.
- **995 have no decision date** (2.7%).
- A valid format doesn't mean a plausible date: placeholder dates (e.g. 01/01/1987
  seen in the download) are a separate check.

## Step 2 · Decision dates before the oldest valid date

The data has no submission date; `valid_date` ("the date at which the application is
valid") is the closest field, and it is on or shortly after submission. The oldest
valid date is the earliest `valid_date` of any Brent application (a `min` aggregation;
`valid_date` is a `date` field, so Elasticsearch compares real dates). A decision can't
come before any application was valid, so a `decision_date` earlier than that is an
anomaly.

In [4]:
body = {
    "size": 0,
    "query": brent_filter(),
    "aggs": {"oldest_valid": {"min": {"field": "valid_date", "format": "dd/MM/yyyy"}}},
}
oldest_valid = search(body)["aggregations"]["oldest_valid"]["value_as_string"]
print("Oldest valid_date in Brent:", oldest_valid)

oldest = pd.to_datetime(oldest_valid, format="%d/%m/%Y")
before = df[parsed < oldest]
print("Decision dates before it:", len(before))
before

Oldest valid_date in Brent: 23/04/1987
Decision dates before it: 25


,id,decision_date
36287,Brent-861288,01/03/1987
36288,Brent-861442,02/01/1987
36289,Brent-861727,01/03/1987
36291,Brent-862186,01/02/1987
36293,Brent-862382,01/02/1987
36294,Brent-862390,01/03/1987
36767,Brent-DY850076,01/01/1987
36768,Brent-DY860443,01/03/1987
36769,Brent-DY860529,01/03/1987
36770,Brent-DY860530,01/03/1987


### Findings · Step 2

- **Oldest `valid_date` in Brent: 23/04/1987.**
- **25 decision dates are earlier than that**, all between 01/01/1987 and 01/04/1987.
- **All 25 are legacy references**: 6 numeric (`861288`, `862186`, ...) and 19 with a
  `DY` prefix (`DY850076`, `DY862014`, ...). The digits look like they start with the
  year (`85`, `86`, `87`).
- **24 of the 25 fall on the 1st of the month** (01/01, 01/02, 01/03 or 01/04/1987);
  the only exception is 02/01/1987. Real decisions don't cluster on the 1st, so these
  look like month-only dates stored as the 1st, not real decision days.
- They are anomalies to investigate: a decision can't come before the oldest
  valid date.

## Step 3 · Decision date before the application's own valid date

For each application that has both dates: is `decision_date` earlier than its own
`valid_date`? A decision can't come before the application was valid, so those are
anomalies. (Same day is allowed.)

In [5]:
rows, after = [], None
while True:
    body = {
        "size": 5000,
        "query": brent_filter(),
        "_source": ["id", "valid_date", "decision_date", "decision"],
        "sort": [{"id": "asc"}],
    }
    if after:
        body["search_after"] = after
    page = search(body)["hits"]["hits"]
    if not page:
        break
    rows += [h["_source"] for h in page]
    after = page[-1]["sort"]

dates = pd.DataFrame(rows).reindex(columns=["id", "valid_date", "decision_date", "decision"])
dates["valid"] = pd.to_datetime(dates["valid_date"], format="%d/%m/%Y", errors="coerce")
dates["decided"] = pd.to_datetime(dates["decision_date"], format="%d/%m/%Y", errors="coerce")

both = dates.dropna(subset=["valid", "decided"])
early = both[both["decided"] < both["valid"]].copy()
early["days_early"] = (early["valid"] - early["decided"]).dt.days

print("Applications with both dates:", len(both), "of", len(dates))
print("Decision before its own valid date:", len(early))

Applications with both dates: 32364 of 36952
Decision before its own valid date: 0


In [6]:
# How early, and the worst cases first.
display(early["days_early"].describe().round(1))
early.sort_values("days_early", ascending=False)[["id", "valid_date", "decision_date", "decision", "days_early"]]

count    0.0
mean     NaN
std      NaN
min      NaN
25%      NaN
50%      NaN
75%      NaN
max      NaN
Name: days_early, dtype: float64

,id,valid_date,decision_date,decision,days_early


### Findings · Step 3

- **32,364 of 36,952** applications have both a `valid_date` and a `decision_date`;
  the other 4,588 miss one or both and can't be checked this way.
- **0 decisions come before their own valid date.** Where both dates exist, they are
  always in the right order.

## Step 4 · The 25 early decision dates: the important columns

The 25 applications from Step 2, fetched by id (an `ids` query, inside the Brent
filter), with the columns the web app shows.

In [7]:
WEB_COLUMNS = [
    "lpa_app_no", "borough", "valid_date", "status", "decision", "decision_date",
    "application_type", "description", "site_name", "site_number", "street_name",
    "locality", "postcode", "decision_target_date", "appeal_decision",
    "appeal_decision_date", "url_planning_app",
]
body = {
    "size": 100,
    "query": {"bool": {"filter": [brent_filter(), {"ids": {"values": before["id"].tolist()}}]}},
    "_source": WEB_COLUMNS,
    "sort": [{"id": "asc"}],
}
early_1987 = pd.DataFrame([h["_source"] for h in search(body)["hits"]["hits"]]).reindex(columns=WEB_COLUMNS)
print("Fetched:", len(early_1987), "of", len(before))
pd.set_option("display.max_colwidth", 60)
early_1987

Fetched: 25 of 25


,lpa_app_no,borough,valid_date,status,decision,decision_date,application_type,description,site_name,site_number,street_name,locality,postcode,decision_target_date,appeal_decision,appeal_decision_date,url_planning_app
0,861288,Brent,None,Lapsed,Approved,01/03/1987,All Other,No description provided by LDMS,KENSAL RISE BAPTIST TABERNACLE,KENSAL RISE BAPTIST TABERNACLE,BULLER ROAD,NaN,NW10,None,NaN,NaN,None
1,861442,Brent,None,Completed,Approved,02/01/1987,All Other,No description provided by LDMS,VACANT LAND NEASDEN SIDINGS,VACANT LAND NEASDEN SIDINGS,GREAT CENTRAL WAY,NaN,NW10,None,NaN,NaN,None
2,861727,Brent,None,Superseded,Approved,01/03/1987,All Other,No description provided by LDMS,NEASDEN HOSPITAL BUILDINGS,NEASDEN HOSPITAL BUILDINGS,BRENTFIELD ROAD,NaN,NW10,None,NaN,NaN,None
3,862186,Brent,None,Completed,Approved,01/02/1987,All Other,No description provided by LDMS,NaN,NaN,COREY,OXGATE CENTRE,NW2,None,NaN,NaN,None
4,862382,Brent,None,Completed,Approved,01/02/1987,All Other,No description provided by LDMS,"VACANT LAND AT, 275","VACANT LAND AT, 275",ABBEYDALE ROAD,NaN,HA0,None,NaN,NaN,None
5,862390,Brent,None,Completed,Approved,01/03/1987,All Other,No description provided by LDMS,VACANT LAND NEASDEN SIDINGS (PART OF),VACANT LAND NEASDEN SIDINGS (PART OF),GREAT CENTRAL WAY,NaN,NW10,None,NaN,NaN,None
6,DY850076,Brent,None,Completed,Approved,01/01/1987,All Other,No description provided by LDMS,SMITHS INDUSTRIES CHRONOS OTBG,SMITHS INDUSTRIES CHRONOS OTBG,NORTH CIRCULAR ROAD,NaN,NW2,None,NaN,NaN,None
7,DY860443,Brent,None,Superseded,Approved,01/03/1987,All Other,No description provided by LDMS,"JESUS & MARY CONVENT SCH.(MAIN BUILDING), 17A","JESUS & MARY CONVENT SCH.(MAIN BUILDING), 17A",CROWNHILL ROAD,NaN,NW10,None,NaN,NaN,None
8,DY860529,Brent,None,Superseded,NaN,01/03/1987,All Other,No description provided by LDMS,13,13,SALMON STREET,NaN,NW9,None,Allowed,01/03/1987,None
9,DY860530,Brent,None,Superseded,Approved,01/03/1987,All Other,No description provided by LDMS,13,13,SALMON STREET,NaN,NW9,None,NaN,NaN,None


### Findings · Step 4

All 25 records look alike: **legacy records with very little detail**.

| Column | What the 25 have |
|---|---|
| `lpa_app_no` | legacy references (`DY85…`–`DY87…`, `86…`), all different |
| `borough` | `Brent` (all) |
| `valid_date` | **empty (all 25)**, so Step 3 couldn't check them |
| `status` | `Completed` 12, `Superseded` 10, `Lapsed` 3 |
| `decision` | `Approved` 23, empty 2 |
| `decision_date` | 01/01–01/04/1987, 24 of them on the 1st of a month |
| `application_type` | `All Other` (all) |
| `description` | **"No description provided by LDMS" (all)** |
| `site_name` | **just a number** (e.g. `13`, `2002`), the same as `site_number` |
| `street_name` | filled, in CAPITALS (e.g. `SALMON STREET` ×5, `EDGWARE ROAD` ×3) |
| `locality` | empty, except 1 (`OXGATE CENTRE`) |
| `postcode` | **only the district** (`NW10`, `NW2`, `NW9`, `HA0`), not a full postcode |
| `decision_target_date` | empty (all) |
| `appeal_decision` | empty, except 2 `Allowed` (the 2 with no `decision`) |
| `appeal_decision_date` | empty, except those 2: 01/03/1987 |
| `url_planning_app` | empty (all) |

- They came from an older system ("LDMS", in the description) and were stored with a
  different address layout: house number in `site_name`, street in `street_name`,
  postcode district only.
- 22 of 25 are `Completed` or `Superseded`: these look like records kept to track what
  was built, not applications you could look up.
- The 2 appeal records (`DY860529`, `DY860634`) have no council `decision`, and their
  `decision_date` and `appeal_decision_date` are the same day (01/03/1987). In reality
  an appeal is decided months after the council's decision, so at least one of those
  dates is not real.